# Final Project for Advanced Data Visualization: Interactive Earthquake Dashboard

Created by Juan Arturo Abaurrea Calafell. Dashboard accessible at https://earthquakes-dashboard.onrender.com/

## Data Description

The earthquake dataset variables are:

- **title**: Name of the title assigned to the earthquake
- **magnitude**: The magnitude of the earthquake
- **date_time**: Date and time
- **cdi**: The maximum reported intensity for the event (range)
- **mmi**: The maximum estimated instrumental intensity for the event
- **alert**: Alert level - "green", "yellow", "orange" and "red"
- **tsunami**: "1" for events in oceanic regions and "0" otherwise
- **sig**: A number describing how significant the event is. Larger numbers indicate a more significant event. This value is determined by several factors, including: magnitude, maximum MMI, felt reports, and estimated impact
- **net**: Data contributor ID. Identifies the network considered as the preferred source of information for this event
- **nst**: The total number of seismic stations used to determine the earthquake location
- **dmin**: Horizontal distance from the epicenter to the nearest station
- **gap**: The largest azimuthal gap between azimuthally adjacent stations (in degrees). In general, the smaller this number, the more reliable the calculated horizontal position of the earthquake. Earthquake locations where the azimuthal gap exceeds 180 degrees typically have large location and depth uncertainties
- **magType**: The method or algorithm used to calculate the preferred magnitude for the event
- **depth**: The depth where the earthquake begins to rupture
- **latitude / longitude**: Coordinate system by which the position or location of any place on Earth's surface can be determined and described
- **location**: Location within the country
- **continent**: Continent of the country where the earthquake struck
- **country**: Affected country

## Dashboard Description

The dashboard is divided into 3 parts:

### 1. General

Statistical analysis and distributions of the main earthquake characteristics:

- **Violin Chart**: Magnitude distribution grouped by alert or magnitude type
- **Density Graph**: Density distribution of magnitude or depth with histogram and KDE
- **Pie/Waffle Chart**: Distribution of alerts or magnitude calculation types
- **Correlation Matrix (Heatmap)**: Correlations between numeric variables (all, magnitude-related, or geographic)
- **Venn Diagram**: Intersections between earthquakes with red alert, tsunamis, and magnitude > 6
- **Bubble Plot**: Scatter plot of relationships between magnitude-depth, magnitude-significance, or depth-significance
- **Radar Chart**: Normalized profile of the selected most significant earthquake from the top 10

### 2. Space

Geographic analysis and spatial distribution of seismic events:

- **Bar Chart / Wordcloud**: Distribution by continent, country, or location (with option to include unknown values)
- **Treemap**: Continent → Country → Location hierarchy with size proportional to number of events
- **Interactive Folium Map**: Visualization with configurable layers (heatmap, marker clusters, circles by magnitude)
- **Area Chart by Latitude**: Distribution of earthquakes by latitude divided into northern and southern hemispheres
- **3D Globe**: Three-dimensional representation of Earth with lines indicating depth, colored by alert or depth
- **Choropleth**: Thematic map of countries colored by selected metric (count, average magnitude, tsunamis, etc.)
- **Box Plot**: Magnitude distribution by continent with outliers marked
- **Machine Learning Prediction**: Random Forest model that predicts the next location and magnitude based on historical events

### 3. Time

Temporal analysis and patterns in the earthquake time series:

- **Time Series Line**: Magnitude or number of earthquakes over time with configurable aggregation (hour, day, month, year) and smoothing
- **Scatter Animation**: Geographic evolution of earthquakes animated by year in 2D or 3D (orthographic projection)
- **Stacked Area Chart**: Evolution of alerts by period (hour, day, month, or year)
- **Calendar Heatmap**: Year vs period (hour/day/month) matrix with event intensity
- **Polar/Circular Chart**: Radial distribution of earthquakes by hour, day, or month of the year

## Preprocesamiento de los datos

Nota: Si hubiera algún problema al ejecutar este notebook, lo más probable es que se deba a incompatibilidades entre versiones de librerías. Esto se puede arreglar utilizando la versión de Python indicada en [.python-version](.python-version) y las versiones de librerías en [requirements.txt](requirements.txt)

In [1]:
import os
import io
import base64
import math
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image
from scipy.stats import gaussian_kde
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn2
import folium
from folium.plugins import HeatMap, MarkerCluster
from wordcloud import WordCloud
from pywaffle import Waffle
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
HOSTING = True # Cambiar a False si se ejecuta localmente
CARPETA_DATASETS = "dataset"
DATASET = "earthquake_1995-2023.csv"
# earthquake_1995-2023.csv tiene los datos de earthquake_data.csv más algunos adicionales
RENDERER = "browser"  # browser o notebook
VALOR_SUSTITUTO_NULO = "desconocido"
COLOR_VALOR_SUSTITUTO_NULO = "gray"

### Cargar y limpiar dataset

In [3]:
df = pd.read_csv(os.path.join(CARPETA_DATASETS, DATASET), encoding='utf-8')

print(df.head(), "\n")

                                      title  magnitude         date_time  cdi  \
0          M 6.5 - 42 km W of Sola, Vanuatu        6.5  16-08-2023 12:47    7   
1  M 6.5 - 43 km S of Intipucá, El Salvador        6.5  19-07-2023 00:22    8   
2  M 6.6 - 25 km ESE of Loncopué, Argentina        6.6  17-07-2023 03:05    7   
3     M 7.2 - 98 km S of Sand Point, Alaska        7.2  16-07-2023 06:48    6   
4                  M 7.3 - Alaska Peninsula        7.3  16-07-2023 06:48    0   

   mmi   alert  tsunami  sig net  nst      dmin    gap magType    depth  \
0    4   green        0  657  us  114  7.177000   25.0     mww  192.955   
1    6  yellow        0  775  us   92  0.679000   40.0     mww   69.727   
2    5   green        0  899  us   70  1.634000   28.0     mww  171.371   
3    6   green        1  860  us  173  0.907000   36.0     mww   32.571   
4    5     NaN        1  820  at   79  0.879451  172.8      Mi   21.000   

   latitude  longitude               location      continent  

In [4]:
total_filas = df.shape[0]

nulos = df.isnull().sum()
nas = df.isna().sum()
vacios = (df == "").sum()

# Filtrar solo columnas con algún valor > 0
mask = (nulos > 0) | (nas > 0) | (vacios > 0)
conteos = pd.DataFrame({
    "Null": nulos[mask],
    "NA": nas[mask],
    "Empty": vacios[mask]
})

porcentajes = pd.DataFrame({
    "Null (%)": (nulos[mask] / total_filas * 100),
    "NA (%)": (nas[mask] / total_filas * 100),
    "Empty (%)": (vacios[mask] / total_filas * 100)
})

print(
    f"Number of rows: {total_filas}\n"
    f"NNumber of columns: {df.shape[1]}\n\n"
    f"Counts of values per column:\n{conteos}\n\n"
    f"Percentage of values per column:\n{porcentajes}"
)

Number of rows: 1000
NNumber of columns: 19

Counts of values per column:
           Null   NA  Empty
alert       551  551      0
location      6    6      0
continent   716  716      0
country     349  349      0

Percentage of values per column:
           Null (%)  NA (%)  Empty (%)
alert          55.1    55.1        0.0
location        0.6     0.6        0.0
continent      71.6    71.6        0.0
country        34.9    34.9        0.0


### Tratamiento de valores desconocidos

Se deducen lugares a partir del lugar más preciso. Por ejemplo: si un terremoto ha sido en Tokyo, el país es Japón, y si el país es Japón, el continente es Asia.

In [5]:
# alert tiene 551 valores nulos/NA
# continent tiene 716 valores nulos/NA
# country tiene 349 valores nulos/NA
# location tiene 6 valores nulos/NA
for col in ["alert", "continent", "country", "location"]:
    df[col] = df[col].fillna(VALOR_SUSTITUTO_NULO)

countries = df["country"].unique().tolist()
countries.remove(VALOR_SUSTITUTO_NULO)

locations_to_fix = df[df["country"] == VALOR_SUSTITUTO_NULO]["location"].unique()
for loc in locations_to_fix:
    if ',' in loc:
        possible_country = loc.split(',')[-1].strip()
        if possible_country and possible_country in countries:
            df.loc[df["location"] == loc, "country"] = possible_country

# mapeos de ubicación a país y de país a continente obtenidos a partir de observar los datos con valores desconocidos
ubicacion_a_pais = {
    'Sand Point, Alaska': 'United States of America',
    'Alaska Peninsula': 'United States of America',
    'Codrington, Antigua and Barbuda': 'Antigua and Barbuda',
    'Tonga': 'Tonga',
    'the Fiji Islands': 'Fiji',
    'the Loyalty Islands': 'New Caledonia',
    'Kermadec Islands region': 'New Zealand',
    'the Kermadec Islands': 'New Zealand',
    'Tadine, New Caledonia': 'New Caledonia',
    'Nikolski, Alaska': 'United States of America',
    'Pólis, Cyprus': 'Cyprus',
    'Vanuatu region': 'Vanuatu',
    'South Sandwich Islands region': 'South Georgia and the South Sandwich Islands',
    'Perryville, Alaska': 'United States of America',
    'Alo, Wallis and Futuna': 'Wallis and Futuna',
    'Lucea, Jamaica': 'Jamaica',
    'Broome, Australia': 'Australia',
    'Bristol Island, South Sandwich Islands': 'South Georgia and the South Sandwich Islands',
    'Olonkinbyen, Svalbard and Jan Mayen': 'Svalbard and Jan Mayen', 
    'Chiniak, Alaska': 'United States of America',
    'Barra Patuca, Honduras': 'Honduras',
    'Attu Station, Alaska': 'United States of America',
    'Komandorskiye Ostrova, Russia region': 'Russia',
    'Auckland Islands, New Zealand region': 'New Zealand',
    'Ferndale, California': 'United States of America',
    'Nicobar Islands, India region': 'India',
    'Fox Islands, Aleutian Islands, Alaska': 'United States of America',
    'Bathsheba, Barbados': 'Barbados',
    'Bonin Islands, Japan region': 'Japan',
    'Ugashik, Alaska': 'United States of America',
    'Fiji region': 'Fiji',
    'Mata-Utu, Wallis and Futuna': 'Wallis and Futuna',
    'Indianola, California': 'United States of America',
    'Okhotsk': 'Russia',
    'Atka, Alaska': 'United States of America',
    'Izu Islands, Japan region': 'Japan',
    'Edna Bay, Alaska': 'United States of America',
    'off the west coast of northern Sumatra': 'Indonesia',
    'Wé, New Caledonia': 'New Caledonia',
    'Matavai, Samoa': 'Samoa',
    'Guanaja, Honduras': 'Honduras',
    'Philippine Islands region': 'Philippines',
    'the Kuril Islands': 'Russia',
    'Kuril Islands': 'Russia',
    'Big Lagoon, California': 'United States of America',
    'Macquarie Island': 'Australia',
    'Old Harbor, Alaska': 'United States of America',
    'Paphos, Cyprus': 'Cyprus',
    'Vao, New Caledonia': 'New Caledonia'
}

pais_a_continente = {
    'Afghanistan': 'Asia',
    'Algeria': 'Africa',
    'Antarctica': 'Antarctica',
    'Argentina': 'South America',
    'Azerbaijan': 'Asia',
    'Bolivia': 'South America',
    'Botswana': 'Africa',
    'Brazil': 'South America',
    'Canada': 'North America',
    'Chile': 'South America',
    'Colombia': 'South America',
    'Costa Rica': 'North America',
    'Ecuador': 'South America',
    'El Salvador': 'North America',
    'Fiji': 'Oceania',
    'Greece': 'Europe',
    'Guatemala': 'North America',
    'Haiti': 'North America',
    'Iceland': 'Europe',
    'India': 'Asia',
    'Indonesia': 'Asia',
    'Iran': 'Asia',
    'Italy': 'Europe',
    'Japan': 'Asia',
    'Kyrgyzstan': 'Asia',
    'Martinique': 'North America',
    'Mexico': 'North America',
    'Mongolia': 'Asia',
    'Mozambique': 'Africa',
    'Myanmar': 'Asia',
    'Nepal': 'Asia',
    'New Zealand': 'Oceania',
    'Nicaragua': 'North America',
    'Pakistan': 'Asia',
    'Panama': 'North America',
    'Papua New Guinea': 'Oceania',
    "People's Republic of China": 'Asia',
    'Peru': 'South America',
    'Philippines': 'Asia',
    'Russia': 'Asia',  # todos los terremotos de Rusia están en la parte asiática
    'Russian Federation (the)': 'Asia',  # todos los terremotos de Rusia están en la parte asiática
    'Saudi Arabia': 'Asia',
    'Solomon Islands': 'Oceania',
    'South Georgia and the South Sandwich Islands': 'Antarctica',
    'Taiwan': 'Asia',
    'Tajikistan': 'Asia',
    'Tanzania': 'Africa',
    'Tonga': 'Oceania',
    'Trinidad and Tobago': 'North America',
    'Turkey': 'Asia',
    'Turkiye': 'Asia',
    'Turkmenistan': 'Asia',
    #'United Kingdom of Great Britain and Northern Ireland (the)': 'Europe', # tiene demasiados territorios en otros continentes
    'United States of America': 'North America',
    'Vanuatu': 'Oceania',
    'Venezuela': 'South America',
    'Barbados': 'North America',
    'Wallis and Futuna': 'Oceania',
    'Jamaica': 'North America',
    'Australia': 'Oceania',
    'Honduras': 'North America',
    'Samoa': 'Oceania',
    'Antigua and Barbuda': 'North America',
    'New Caledonia': 'Oceania',
    'Cyprus': 'Europe',
    'Svalbard and Jan Mayen': 'Europe',
    VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
}

df.loc[df['country'] == VALOR_SUSTITUTO_NULO, 'country'] = df.loc[df['country'] == VALOR_SUSTITUTO_NULO, 'location'].map(ubicacion_a_pais).fillna(VALOR_SUSTITUTO_NULO)
df.loc[df['continent'] == VALOR_SUSTITUTO_NULO, 'continent'] = df.loc[df['continent'] == VALOR_SUSTITUTO_NULO, 'country'].map(pais_a_continente).fillna(VALOR_SUSTITUTO_NULO)

### Traducción de lugares y alertas

In [6]:
# Translation toggle
ENABLE_TRANSLATION = False  # Set to False for English version

if ENABLE_TRANSLATION:
    # Spanish translations
    traduccion_colores = {
        "green": "verde",
        "yellow": "amarilla",
        "red": "roja",
        "orange": "naranja",
        VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
    }

    df["alert"] = df["alert"].map(traduccion_colores)

    # mapping that relates alert name with its color
    colores_alerta = {v: k for k, v in traduccion_colores.items()}

    traduccion_continentes = {
        "Asia": "Asia",
        "North America": "Norteamérica",
        "South America": "Sudamérica",
        "Europe": "Europa",
        "Africa": "África",
        "Oceania": "Oceanía",
        "Antarctica": "Antártida",
        VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
    }

    df["continent"] = df["continent"].map(traduccion_continentes)

    traduccion_paises = {
        'Afghanistan': 'Afganistán',
        'Algeria': 'Argelia',
        'Antarctica': 'Antártida',
        'Argentina': 'Argentina',
        'Azerbaijan': 'Azerbaiyán',
        'Bolivia': 'Bolivia',
        'Botswana': 'Botsuana',
        'Brazil': 'Brasil',
        'Canada': 'Canadá',
        'Chile': 'Chile',
        'Colombia': 'Colombia',
        'Costa Rica': 'Costa Rica',
        'Ecuador': 'Ecuador',
        'El Salvador': 'El Salvador',
        'Fiji': 'Fiyi',
        'Greece': 'Grecia',
        'Guatemala': 'Guatemala',
        'Haiti': 'Haití',
        'Iceland': 'Islandia',
        'India': 'India',
        'Indonesia': 'Indonesia',
        'Iran': 'Irán',
        'Italy': 'Italia',
        'Japan': 'Japón',
        'Kyrgyzstan': 'Kirguistán',
        'Martinique': 'Martinica',
        'Mexico': 'México',
        'Mongolia': 'Mongolia',
        'Mozambique': 'Mozambique',
        'Myanmar': 'Myanmar',
        'Nepal': 'Nepal',
        'New Zealand': 'Nueva Zelanda',
        'Nicaragua': 'Nicaragua',
        'Pakistan': 'Pakistán',
        'Panama': 'Panamá',
        'Papua New Guinea': 'Papúa Nueva Guinea',
        "People's Republic of China": 'República Popular China',
        'Peru': 'Perú',
        'Philippines': 'Filipinas',
        'Russia': 'Rusia',
        'Russian Federation (the)': 'Federación de Rusia',
        'Saudi Arabia': 'Arabia Saudí',
        'Solomon Islands': 'Islas Salomón',
        'South Georgia and the South Sandwich Islands': 'Islas Georgias del Sur y Sandwich del Sur',
        'Taiwan': 'Taiwán',
        'Tajikistan': 'Tayikistán',
        'Tanzania': 'Tanzania',
        'Tonga': 'Tonga',
        'Trinidad and Tobago': 'Trinidad y Tobago',
        'Turkey': 'Turquía',
        'Turkiye': 'Turquía',
        'Turkmenistan': 'Turkmenistán',
        'United Kingdom of Great Britain and Northern Ireland (the)': 'Reino Unido de Gran Bretaña e Irlanda del Norte',
        'United States of America': 'Estados Unidos de América',
        'Vanuatu': 'Vanuatu',
        'Venezuela': 'Venezuela',
        'Barbados': 'Barbados',
        'Wallis and Futuna': 'Wallis y Futuna',
        'Jamaica': 'Jamaica',
        'Australia': 'Australia',
        'Honduras': 'Honduras',
        'Samoa': 'Samoa',
        'Antigua and Barbuda': 'Antigua y Barbuda',
        'New Caledonia': 'Nueva Caledonia',
        'Cyprus': 'Chipre',
        'Svalbard and Jan Mayen': 'Svalbard y Jan Mayen',
        VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
    }

    df["country"] = df["country"].map(traduccion_paises)

    posiciones_alerta = {
        "verde": 0,
        "amarilla": 1,
        "naranja": 2,
        "roja": 3,
        VALOR_SUSTITUTO_NULO: 4
    }

else:
    # English version - no translation needed
    # Keep original alert values
    colores_alerta = {
        "green": "green",
        "yellow": "yellow",
        "orange": "orange",
        "red": "red",
        VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
    }

    # Continents remain in English (already in English in source data)
    # No translation needed for continents
    
    # Countries remain in English (already in English in source data)
    # No translation needed for countries

    posiciones_alerta = {
        "green": 0,
        "yellow": 1,
        "orange": 2,
        "red": 3,
        VALOR_SUSTITUTO_NULO: 4
    }

### Creación de constantes

Estas contienen todos los continentes, países, alertas y años, además del año mínimo y máximo

In [7]:
df["date_time"] = pd.to_datetime(df["date_time"], format="%d-%m-%Y %H:%M")
df['year'] = df['date_time'].dt.year

CONTINENTS = sorted([c for c in df['continent'].unique() if c != VALOR_SUSTITUTO_NULO]) + [VALOR_SUSTITUTO_NULO]
COUNTRIES = sorted([c for c in df['country'].unique() if c != VALOR_SUSTITUTO_NULO]) + [VALOR_SUSTITUTO_NULO]
ALERTS = sorted([c for c in df['alert'].unique()], key=lambda x: posiciones_alerta[x])
YEARS = sorted([int(y) for y in df['year'].unique()])
YEAR_MIN, YEAR_MAX = min(YEARS), max(YEARS)

### Definición de métodos para cargar la textura de la Tierra

In [8]:
def load_earth_texture_rgb(image_path, resolution=(100, 100)):
    """Carga la textura de la Tierra preservando los colores RGB"""
    img = Image.open(image_path)
    img = img.convert("RGB")
    img = img.resize(resolution)
    img_array = np.array(img)
    return img_array

def create_colorscale_from_image(img_array):
    """Crea una escala de color que preserva los colores de la imagen"""
    
    gray = 0.299 * img_array[:,:,0] + 0.587 * img_array[:,:,1] + 0.114 * img_array[:,:,2]
    
    n_samples = 256
    colorscale = []
    
    for i in range(n_samples):
        level = i * 255 / (n_samples - 1)
        # Encontrar el píxel más cercano a este nivel de brillo
        diff = np.abs(gray - level)
        idx = np.unravel_index(np.argmin(diff), gray.shape)
        r, g, b = img_array[idx[0], idx[1], :]
        colorscale.append([i / (n_samples - 1), f'rgb({r},{g},{b})'])
    
    return colorscale, gray

## Creación del propio dashboard y sus callbacks

In [9]:
# Preparar datos para el radar chart (top 10 terremotos)
df_top10 = df.nlargest(10, 'magnitude').reset_index(drop=True)

# -------------------------
# Funciones auxiliares
# -------------------------

def filter_df(continent, country, year_range, only_tsunami, alerts_selected):
    d = df.copy()
    if continent and continent != 'All':
        d = d[d['continent'] == continent]
    if country and country != 'All':
        d = d[d['country'] == country]
    if year_range:
        start, end = year_range
        d = d[(d['year'] >= start) & (d['year'] <= end)]
    if only_tsunami:
        d = d[d['tsunami'] == 1]
    if alerts_selected and 'All' not in alerts_selected:
        d = d[d['alert'].isin(alerts_selected)]
    return d

def kpi_card(title, value, subtitle="", help_text=None):
    if help_text:
        title_content = html.Span([
            title,
            html.Span(
                "ⓘ",
                title=help_text,
                style={
                    'cursor': 'help',
                    'fontSize': '0.9em',
                    'color': '#6c757d',
                    'fontWeight': 'bold',
                    'marginLeft': '4px'  # small optional margin
                }
            )
        ])
    else:
        title_content = title

    return dbc.Card(
        dbc.CardBody([
            html.H6(title_content, className="card-title text-muted"),
            html.H4(value, className="card-value"),
            html.Div(subtitle, className="card-subtitle text-muted")
        ]), className="h-100"
    )

# -------------------------
# App Layout
# -------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server

column_style = {
    'backgroundColor': '#ffffff',
    'padding': '0.5em 1em',
    'borderRadius': '0.5em',
    #'height': '30rem'
}

sidebar = dbc.Card([
    html.H5("Filters", className="mt-2 mb-2"),
    dbc.Label("Continent"),
    dcc.Dropdown(options=['All'] + CONTINENTS, value='All', id='filter-continent', clearable=False),
    dbc.Label("Country", className='mt-2'),
    dcc.Dropdown(options=['All'] + COUNTRIES, value='All', id='filter-country', clearable=False),
    dbc.Label("Years", className='mt-2'),
    dcc.RangeSlider(
        id='filter-year',
        min=YEAR_MIN,
        max=YEAR_MAX,
        step=1,
        value=[max(YEAR_MIN, YEAR_MAX - 4), YEAR_MAX],
        marks={YEAR_MIN: str(YEAR_MIN), YEAR_MAX: str(YEAR_MAX)}
    ),
    html.Div(id='year-selected', className='mt-2 text-center'),
    dbc.Checklist(options=[{"label": "Tsunamis only", "value": "tsunami"}], value=[], id='filter-tsunami', inline=True, className='mt-3'),
    dbc.Label("Alerts", className='mt-2'),
    dcc.Dropdown(options=ALERTS, value=ALERTS, id='filter-alerts', multi=True),
    dbc.Button("Apply filters", id='apply-filters', color='primary', className='me-3 mt-3', n_clicks=0),
    dbc.Button("Reset", id='reset-filters', color='secondary', className='mt-3', n_clicks=0)
], body=True)

footer = dbc.Container([
    html.Hr(className='my-4'),
    dbc.Row([
        dbc.Col([
            html.H5("About the Dashboard", className='mb-3'),
            html.P([
                "Interactive dashboard for worldwide earthquake analysis. ",
                "Developed to visualize and explore seismic data in an intuitive way."
            ]),
            html.P([
                html.Strong("Author: "),
                "Juan Arturo Abaurrea Calafell",
                html.Br(),
                html.Strong("Contact: "),
                html.A("LinkedIn", href="https://www.linkedin.com/in/j-arturo-a-c", target="_blank"),
                " | ",
                html.A("GitHub", href="https://github.com/Jarturog", target="_blank"),
                html.Br(),
                html.Strong("Source code: "),
                html.A("GitHub", href="https://github.com/Jarturog/earthquakes-dashboard", target="_blank")
            ]),
            html.P([
                html.Small("Last updated: November 2025", className='text-muted')
            ])
        ], width=12, md=4),
        

        dbc.Col([
            html.H5("Libraries", className='mb-3'),
            html.Ul([
                html.Li([html.Strong("Pandas & NumPy"), " - Data processing and analysis"]),
                html.Li([html.Strong("Dash"), " - Dashboard framework"]),
                html.Li([html.Strong("Matplotlib, PyWaffle, WordCloud"), " - Static graphics"]),
                html.Li([html.Strong("Plotly"), " - Interactive graphics"]),
                html.Li([html.Strong("Folium"), " - Interactive map"]),
                html.Li([html.Strong("Scikit-learn"), " - Machine Learning modeling and prediction"]),
                html.Li([html.Strong("PIL"), " - Earth map processing for 3D globe"]),
            ], className='mb-0')
        ], width=12, md=4),

        dbc.Col([
            dbc.Row([
                dbc.Col([
                    html.H5("Data Source", className='mb-3'),
                    html.P([
                        html.Strong("Dataset: "),
                        html.A(
                            "Earthquake dataset",
                            href="https://www.kaggle.com/datasets/warcoder/earthquake-dataset/data",
                            target="_blank"
                        ),
                    ]),
                    html.P([
                        "The data comes from Kaggle."
                    ])
                ], width=6, md=6),

                dbc.Col([
                    html.H6("Future expansions", className='mb-3'),
                    html.P([
                        "The dashboard could be expanded by incorporating volcano data to complement the geological and seismic analysis. ",
                        "For example, using the dataset ",
                        html.A(
                            "Volcanoes on Earth (2021)",
                            href="https://www.kaggle.com/datasets/ramjasmaurya/volcanoes-on-earth-in-2021",
                            target="_blank"
                        ),
                        " available on Kaggle."
                    ])
                ], width=6, md=6)
            ])
        ], width=12, md=4)
    ]),
    html.Hr(className='my-3'),
    dbc.Row([
        dbc.Col([
            html.P([
                "2025 - Earthquake Dashboard | ",
                html.Small("Developed for the Advanced Data Visualization course of the Master's Degree in Machine Learning and Massive Data at the Universidad Politécnica de Madrid")
            ], className='text-center text-muted mb-0')
        ])
    ])
], fluid=True, className='mt-5 mb-3')

def radar_placeholder():
    fig = go.Figure()
    fig.update_layout(height=300, title='Select an earthquake to view radar')
    return fig

radar_fig = radar_placeholder()

tab_general_content = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            html.H5('Magnitude Distribution by: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
            dcc.RadioItems(
                id='violin-groupby',
                options=[
                    {'label': 'Alert', 'value': 'alert'},
                    {'label': 'Magnitude Type', 'value': 'magType'}
                ],
                value='alert',
                inline=True,
                labelStyle={'margin-right': '1.5em', 'whiteSpace': 'nowrap'},
                className='mb-0'
            ),
            dcc.Graph(id='violin-graph')
        ], style=column_style), width=4),
        dbc.Col(html.Div([
            html.H5('Distribution of: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
            dcc.RadioItems(
                id='density-variable',
                options=[{'label': 'Magnitude', 'value': 'magnitude'}, {'label': 'Depth', 'value': 'depth'}],
                value='magnitude',
                inline=True,
                labelStyle={'margin-right': '1.5em', 'whiteSpace': 'nowrap'},
                className='mb-0'
            ),
            dcc.Graph(id='density-graph')
        ], style=column_style), width=4),
        dbc.Col(html.Div([
            html.H5('Distribution of: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
            dcc.RadioItems(
                id='pie-variable',
                options=[
                    {'label': 'Alerts', 'value': 'alert'},
                    {'label': 'Magnitude Calculation Types', 'value': 'magType'}
                ],
                value='alert',
                inline=True,
                labelStyle={'margin-right': '1.5em', 'whiteSpace': 'nowrap'},
                className='mb-0'
            ),
            dcc.Dropdown(
                id='pie-chart-type',
                options=[{'label': 'Pie Chart', 'value': 'pie'}, {'label': 'Waffle Chart', 'value': 'waffle'}],
                value='pie',
                clearable=False,
                className='mb-2'
            ),
            dcc.Graph(id='pie-waffle-graph')
        ], style=column_style), width=4)
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(html.Div([
            html.Div([
                html.H5('Correlation Matrix: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                dcc.Dropdown(
                    id='heatmap-vars',
                    options=[
                        {'label': 'All variables', 'value': 'all'},
                        {'label': 'Magnitude and related', 'value': 'magnitude_related'},
                        {'label': 'Geographic', 'value': 'geographic'}
                    ],
                    value='all',
                    clearable=False,
                    className='mb-0',
                    style={'minWidth': '200px'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            dcc.Graph(id='heatmap-graph')
        ], style=column_style), width=8),
        dbc.Col(html.Div([
            html.H5('Venn Diagram', className='mb-2'),
            html.Img(id='venn-diagram', style={'width': '100%', 'height': 'auto'})
        ], style=column_style), width=4)
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(html.Div([
            html.Div([
                html.H5('Bubble Plot: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                dcc.Dropdown(
                    id='bubble-axes',
                    options=[
                        {'label': 'Magnitude vs Depth', 'value': 'mag_depth'},
                        {'label': 'Magnitude vs Significance', 'value': 'mag_sig'},
                        {'label': 'Depth vs Significance', 'value': 'depth_sig'}
                    ],
                    value='mag_depth',
                    clearable=False,
                    className='mb-0',
                    style={'width': '75%'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            dcc.Graph(id='bubble-graph')
        ], style=column_style), width=7),
        dbc.Col(html.Div([
            html.H5('Radar of Top 10 Earthquakes'), 
            dcc.Dropdown(
                id='radar-select',
                options=[{'label': r, 'value': i} for i, r in enumerate(df_top10['title'])],
                value=0,
                style={'whiteSpace': 'nowrap'}
            ), 
            dcc.Graph(id='radar-graph', figure=radar_fig)
        ], style=column_style), width=5)
    ], className='mb-3')
], id='tab-general-content', style={'display': 'none'})
    
tab_espacio_content = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            html.Div([
                html.H5('Distribution by: ', className='me-3 mb-0'),
                dcc.Dropdown(
                    id='espacio-bar-variable',
                    options=[
                        {'label': 'Continent', 'value': 'continent'},
                        {'label': 'Country', 'value': 'country'},
                        {'label': 'Location', 'value': 'location'}
                    ],
                    value='continent',
                    clearable=False,
                    style={'width': '200px'}
                ),
                html.Span("Include Unknown Values:", className='ms-3 me-1'),
                dcc.Checklist(
                    id='espacio-bar-desconocido',
                    options=[{'label': '', 'value': True}],
                    value=[],
                    inline=True
                )
            ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '10px'}),
            dcc.RadioItems(
                id='espacio-bar-type',
                options=[
                    {'label': 'Bar Chart', 'value': 'bar'},
                    {'label': 'Word Cloud', 'value': 'wordcloud'}
                ],
                value='wordcloud',
                inline=True,
                labelStyle={'margin-right': '1.5em'}
            ),
            dcc.Graph(id='espacio-bar-graph')
        ], style=column_style), width=6),
        
        dbc.Col(html.Div([
            html.H5('Treemap: Continent → Country → Location'),
            dcc.Graph(id='espacio-treemap-graph')
        ], style=column_style), width=6)
    ], className='mb-3'),
    
    dbc.Row([
        dbc.Col(html.Div([
            html.Div([
                html.H5('Interactive Folium Map: ', className='me-3 mb-0'),
                dcc.Dropdown(
                    id='espacio-folium-basemap',
                    options=[
                        {'label': 'OpenStreetMap', 'value': 'OpenStreetMap'},
                        {'label': 'CartoDB Positron', 'value': 'CartoDB positron'},
                        {'label': 'CartoDB Dark', 'value': 'CartoDB dark_matter'}
                    ],
                    value='CartoDB dark_matter',
                    clearable=False,
                    style={'width': '200px'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '10px'}),
            html.Div([
                dbc.Checklist(
                    id='espacio-folium-layers',
                    options=[
                        {'label': 'Heatmap', 'value': 'heatmap'},
                        {'label': 'Clustered Markers', 'value': 'clusters'},
                        {'label': 'Circles by Magnitude', 'value': 'circles'}
                    ],
                    value=['heatmap'],
                    inline=True,
                    className='mb-2'
                ),
            ]),
            html.Iframe(id='espacio-folium-map', style={'width': '100%', 'height': '450px', 'border': 'none'})
        ], style=column_style), width=8),

        dbc.Col(html.Div([
            html.H5('Earthquake Distribution by Latitude'),
            dcc.Graph(id='espacio-latitude-area-graph')
        ], style=column_style), width=4)
    ], className='mb-3'),
    
    dbc.Row([
        dbc.Col(
            html.Div([
                html.H5("3D Globe with Earthquakes", className="mb-1 fw-bold"),
                html.P(
                    "Thickness indicates magnitude.",
                    className="text-muted mb-1"
                ),
                html.Div([
                    html.Label("Color by:", className="me-2 fw-semibold"),
                    dcc.RadioItems(
                        id="espacio-3d-color",
                        options=[
                            {"label": "Alert", "value": "alert"},
                            {"label": "Depth", "value": "depth"}
                        ],
                        value="alert",
                        inline=True,
                        labelStyle={"margin-right": "1em"}
                    ),
                    html.Span("·", className="mx-2 text-muted"),
                    html.Label("Scale Factor:", className="me-2 fw-semibold"),
                    dcc.Input(
                        id='espacio-3d-factor',
                        type='number', min=1, max=5, step=1, value=3,
                        className='form-control d-inline-block',
                        style={'width': '80px', 'display': 'inline-block'}
                    ),
                ], className="d-flex align-items-center flex-wrap mb-3"),
                dcc.Graph(id='espacio-3d-globe-graph')
            ], style=column_style),
            width=6
        ),
        
        dbc.Col(html.Div([
            html.Div([
                html.H5('Choropleth by: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                dcc.Dropdown(
                    id='espacio-choropleth-metric',
                    options=[
                        {'label': 'Number of Earthquakes', 'value': 'count'},
                        {'label': 'Average Magnitude', 'value': 'mean_mag'},
                        {'label': 'Maximum Magnitude', 'value': 'max_mag'},
                        {'label': 'Average Significance', 'value': 'mean_sig'},
                        {'label': 'Number of Tsunamis', 'value': 'tsunami_count'},
                        {'label': 'Average Alert', 'value': 'mean_alert'},
                        {'label': 'Average MMI', 'value': 'mean_mmi'}
                    ],
                    value='count',
                    clearable=False,
                    style={'width': '75%'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '10px'}),
            dcc.Graph(id='espacio-choropleth-graph')
        ], style=column_style), width=6)
    ], className='mb-3'),

    dbc.Row([
        dbc.Col(html.Div([
            html.H5('Magnitude Distribution by Continent'),
            dcc.Graph(id='espacio-boxplot-graph')
        ], style=column_style), width=4),

        dbc.Col(html.Div([
            html.H5('Next Earthquake Prediction (Random Forest)'),
            dbc.Button("Train Model and Predict", id='espacio-predict-button', 
                    color='primary', className='mb-3'),
            html.Div(id='espacio-prediction-results'),
            dcc.Graph(id='espacio-prediction-map')
        ], style=column_style), width=8)
    ], className='mb-3')
], id='tab-espacio-content', style={'display': 'block'})

tab_tiempo_content = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            html.Div([
                html.H5('Magnitude over Time. Group by: ', className='me-3 mb-0'),
                dcc.RadioItems(
                    id='time-line-period',
                    options=[
                        {'label': 'None', 'value': ''},
                        {'label': 'Hour', 'value': 'H'},
                        {'label': 'Day', 'value': 'D'},
                        {'label': 'Month', 'value': 'ME'},
                        {'label': 'Year', 'value': 'YE'}
                    ],
                    value='',
                    inline=True,
                    labelStyle={'margin-right': '1.5em'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            html.Div([
                html.H6('Moving average smoothing parameter: ', className='me-3 mb-0'),
                dcc.Input(id='time-line-smoothing', type='number', min=1, max=30, step=1, value=1, className='ms-3'),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            dcc.Graph(id='time-line-graph')
        ], style=column_style), width=12)
    ], className='mb-3'),
    
    dbc.Row([
        dbc.Col(html.Div([
            html.H5('Geographic Evolution by Year'),
            dcc.RadioItems(
                id='time-animation-dimension',
                options=[
                    {'label': '2D', 'value': 2},
                    {'label': '3D', 'value': 3}
                ],
                value=2,
                inline=True,
                labelStyle={'margin-right': '1.5em'}
            ),
            dcc.Graph(id='time-animation-graph')
        ], style=column_style), width=6),
        
        dbc.Col(html.Div([
            html.Div([
                html.H5('Alert Evolution by: ', className='me-3 mb-0'),
                dcc.RadioItems(
                    id='alert-timeseries-period',
                    options=[
                        {'label': 'Hour', 'value': 'H'},
                        {'label': 'Day', 'value': 'D'},
                        {'label': 'Month', 'value': 'M'},
                        {'label': 'Year', 'value': 'Y'}
                    ],
                    value='Y',
                    inline=True,
                    labelStyle={'margin-right': '1.5em'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            dcc.Graph(id='alert-timeseries-graph')
        ], style=column_style), width=6)
    ], className='mb-3'),
    
    dbc.Row([
        dbc.Col(html.Div([
            html.Div([
                html.H5('Earthquake Calendar by: ', className='me-3 mb-0'),
                dcc.RadioItems(
                    id='calendar-granularity',
                    options=[
                        {'label': 'Hour', 'value': 'hour'},
                        {'label': 'Day', 'value': 'day'},
                        {'label': 'Month', 'value': 'month'}
                    ],
                    value='month',
                    inline=True,
                    labelStyle={'margin-right': '1.5em'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            dcc.Graph(id='calendar-heatmap-graph')
        ], style=column_style), width=6),
        
        dbc.Col(html.Div([
            html.Div([
                html.H5('Circular Distribution by: ', className='me-3 mb-0'),
                dcc.RadioItems(
                    id='polar-type',
                    options=[
                        {'label': 'Hour', 'value': 'hour'},
                        {'label': 'Day', 'value': 'day'},
                        {'label': 'Month', 'value': 'month'}
                    ],
                    value='hour',
                    inline=True,
                    labelStyle={'margin-right': '1.5em'}
                ),
            ], style={'display': 'flex', 'alignItems': 'center'}),
            dcc.Graph(id='polar-chart-graph')
        ], style=column_style), width=6)
    ], className='mb-3')
], id='tab-tiempo-content', style={'display': 'none'})

layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Earthquake Dashboard"), width=10)
    ], align='center', justify='center', className='mb-3 mt-4'),

    dbc.Row([
        dbc.Col([
            sidebar,
            html.Div(id='kpi-col', className='mt-3')
        ], width=2),

        dbc.Col([
            dcc.Tabs(id='tabs', value='tab-general', children=[
                dcc.Tab(label='General', value='tab-general'),
                dcc.Tab(label='Space', value='tab-espacio'),
                dcc.Tab(label='Time', value='tab-tiempo'),
            ]),
            html.Div([
                tab_general_content,
                tab_espacio_content,
                tab_tiempo_content
            ])
        ], width=8),
    ], justify='center'),
    
    footer
    
], style={'background': "#f0f0f4"}, fluid=True)

app.layout = layout

# -------------------------
# Callbacks
# -------------------------
@app.callback(
    Output('year-selected', 'children'),
    Input('filter-year', 'value')
)
def update_year_label(selected_range):
    return f"{selected_range[0]} - {selected_range[1]}"

@app.callback(
    Output('kpi-col', 'children'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_kpis(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    total = len(d)
    
    # Basic calculations
    avg_mag = round(d['magnitude'].mean(), 2) if total > 0 else 'N/A'
    max_mag = round(d['magnitude'].max(), 2) if total > 0 else 'N/A'
    min_mag = round(d['magnitude'].min(), 2) if total > 0 else 'N/A'
    
    # Tsunamis
    num_tsunamis = d['tsunami'].sum() if total > 0 and 'tsunami' in d.columns else 0
    pct_tsu = f"{round(100 * (num_tsunamis / total), 1)}%" if total > 0 else '0%'
    
    # Depth
    avg_depth = round(d['depth'].mean(), 1) if total > 0 and 'depth' in d.columns else 'N/A'
    max_depth = round(d['depth'].max(), 1) if total > 0 and 'depth' in d.columns else 'N/A'
    
    # Significance
    avg_sig = round(d['sig'].mean(), 0) if total > 0 and 'sig' in d.columns else 'N/A'
    max_sig = round(d['sig'].max(), 0) if total > 0 and 'sig' in d.columns else 'N/A'
    
    # Alerts
    alertas_dist = d['alert'].value_counts().to_dict() if 'alert' in d.columns else {}
    num_rojas = alertas_dist.get('red', 0)
    num_naranjas = alertas_dist.get('orange', 0)
    num_amarillas = alertas_dist.get('yellow', 0)
    num_verdes = alertas_dist.get('green', 0)

    # Critical alerts: red + orange
    pct_criticas = f"{round(100 * (num_rojas + num_naranjas) / total, 1)}%" if total > 0 else '0%'

    # Moderate alerts: merge yellow and green
    num_moderadas = num_amarillas + num_verdes
    pct_moderadas = f"{round(100 * num_moderadas / total, 1)}%" if total > 0 else '0%'

    # New statistic: Earthquakes with magnitude >= 7
    num_mag7 = len(d[d['magnitude'] >= 7]) if total > 0 else 0
    pct_mag7 = f"{round(100 * num_mag7 / total, 1)}%" if total > 0 else '0%'

    # Intensities
    avg_mmi = round(d['mmi'].mean(), 1) if total > 0 and 'mmi' in d.columns and d['mmi'].notna().any() else 'N/A'
    avg_cdi = round(d['cdi'].mean(), 1) if total > 0 and 'cdi' in d.columns and d['cdi'].notna().any() else 'N/A'
    
    # Most affected countries
    if 'country' in d.columns and total > 0:
        top_country = d[d['country'] != VALOR_SUSTITUTO_NULO]['country'].value_counts().head(1)
        if len(top_country) > 0:
            pais_mas_afectado = f"{top_country.index[0]} ({top_country.values[0]})"
        else:
            pais_mas_afectado = 'N/A'
    else:
        pais_mas_afectado = 'N/A'
    
    # Most affected continents
    if 'continent' in d.columns and total > 0:
        top_continent = d[d['continent'] != VALOR_SUSTITUTO_NULO]['continent'].value_counts().head(1)
        if len(top_continent) > 0:
            continente_mas_afectado = f"{top_continent.index[0]} ({top_continent.values[0]})"
        else:
            continente_mas_afectado = 'N/A'
    else:
        continente_mas_afectado = 'N/A'
    
    # Temporal trend (if there is data from different years)
    if 'year' in d.columns and total > 0:
        years_data = d.groupby('year', observed=True).size()
        if len(years_data) > 1:
            trend = "↑" if years_data.iloc[-1] > years_data.iloc[0] else "↓"
            trend_text = f"{trend} {abs(round(((years_data.iloc[-1] - years_data.iloc[0]) / years_data.iloc[0]) * 100, 1))}%"
        else:
            trend_text = "N/A"
    else:
        trend_text = "N/A"
    
    # Most significant earthquake
    if total > 0:
        most_sig = d.nlargest(1, 'magnitude').iloc[0]
        terremoto_principal = f"{most_sig['title']}"
    else:
        terremoto_principal = 'N/A'

    cards = dbc.Row([
        dbc.Col(kpi_card('Total Earthquakes', total, [f'Trend:', html.Br(), f'{trend_text}']), width=6, className='mb-2'),
        dbc.Col(kpi_card('Average Magnitude', avg_mag, [f'Range:', html.Br(), f'{min_mag} - {max_mag}']), width=6, className='mb-2'),

        dbc.Col(kpi_card('Tsunamis', [f'{num_tsunamis}', html.Br(), f'({pct_tsu})'], 'of total events'), width=6, className='mb-2'),
        dbc.Col(kpi_card('Earthquakes ≥ M7', f'{num_mag7}', f'({pct_mag7})'), width=6, className='mb-2'),
        
        dbc.Col(kpi_card('Critical Alerts', pct_criticas, [f'🔴 {num_rojas}', html.Br(), f'🟠 {num_naranjas}']), width=6, className='mb-2'),
        dbc.Col(kpi_card('Moderate Alerts', pct_moderadas, [f'🟡 {num_amarillas}', html.Br(), f'🟢 {num_verdes}']), width=6, className='mb-2'),

        dbc.Col(kpi_card('Average Depth', f'{avg_depth} km', [f'Maximum:', html.Br(), f'{max_depth} km']), width=6, className='mb-2'),
        dbc.Col(kpi_card(
            'Average Significance', 
            avg_sig, 
            [f'Maximum:', html.Br(), f'{max_sig}'],
            help_text='Number describing the importance of the event. Higher values indicate more significant events. Determined by factors such as: magnitude, maximum MMI, perceived reports, and estimated impact.'
        ), width=6, className='mb-2'),
        
        dbc.Col(kpi_card(
            'Average MMI', 
            avg_mmi, 
            'Instrumental intensity',
            help_text='Modified Mercalli Intensity: Scale that measures earthquake intensity based on seismic instruments (1-12)'
        ), width=6, className='mb-2'),
        dbc.Col(kpi_card(
            'Average CDI', 
            avg_cdi, 
            'Reported intensity',
            help_text='Community Decimal Intensity: Intensity perceived and reported by the affected population (1-10)'
        ), width=6, className='mb-2'),

        dbc.Col(kpi_card(
            'Most Affected Country', 
            pais_mas_afectado, 
            '',
            help_text='Country with the highest number of earthquakes recorded in the selected period. The number in parentheses indicates the number of events.'
        ), width=6, className='mb-2'),
        dbc.Col(kpi_card(
            'Most Affected Continent', 
            continente_mas_afectado, 
            '',
            help_text='Continent with the highest seismic activity in the selected period. The number in parentheses indicates the number of recorded events.'
        ), width=6, className='mb-2'),

        dbc.Col(kpi_card('Earthquake With Highest Magnitude', terremoto_principal, f'Magnitude: {max_mag}'), width=12, className='mb-2'),
    ], className='g-2 align-items-stretch')

    return cards


@app.callback(
    [Output('tab-general-content', 'style'),
     Output('tab-espacio-content', 'style'),
     Output('tab-tiempo-content', 'style')],
    Input('tabs', 'value')
)
def render_tab(tab):
    if tab == 'tab-general':
        return {'display': 'block'}, {'display': 'none'}, {'display': 'none'}
    elif tab == 'tab-espacio':
        return {'display': 'none'}, {'display': 'block'}, {'display': 'none'}
    elif tab == 'tab-tiempo':
        return {'display': 'none'}, {'display': 'none'}, {'display': 'block'}
    else:
        return {'display': 'none'}, {'display': 'block'}, {'display': 'none'}

@app.callback(
    Output('violin-graph', 'figure'),
    Input('violin-groupby', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_violin(groupby_var, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if groupby_var not in d.columns or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text=f"Insufficient data", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    fig = px.violin(
        d,
        y='magnitude',
        x=groupby_var,
        box=True,
        points=None,
        color=groupby_var,
        color_discrete_map=colores_alerta if groupby_var == 'alert' else None,
        category_orders={'alert': ALERTS} if groupby_var == 'alert' else None
    )
    fig.update_layout(
        height=350,
        xaxis_title=groupby_var.capitalize(),
        yaxis_title='Magnitude',
        showlegend=True,
        legend=dict(
            orientation="h",  # Horizontal legend
            yanchor="bottom",
            y=1.02,  # Position above the graph
            xanchor="center",
            x=0.5
        ),
        legend_title_text='',
        margin=dict(l=50, r=20, t=40, b=50)  # Tighter margins
    )
    return fig

@app.callback(
    Output('heatmap-graph', 'figure'),
    Input('heatmap-vars', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_heatmap(var_selection, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    numeric_cols = d.select_dtypes(include='number').columns.tolist()
    
    if var_selection == 'magnitude_related':
        cols = [c for c in ['magnitude', 'sig', 'cdi', 'mmi', 'dmin', 'gap'] if c in numeric_cols]
    elif var_selection == 'geographic':
        cols = [c for c in ['latitude', 'longitude', 'depth'] if c in numeric_cols]
    else:  # 'all'
        cols = numeric_cols
    
    if len(cols) < 2:
        fig = go.Figure()
        fig.add_annotation(text="Not enough numeric variables", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    corr_matrix = d[cols].corr()
    
    fig = px.imshow(
        corr_matrix,
        text_auto='.2f',
        aspect='auto',
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1
    )
    fig.update_layout(height=350)
    return fig

@app.callback(
    Output('bubble-graph', 'figure'),
    Input('bubble-axes', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_bubble(axes_selection, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    axes_map = {
        'mag_depth': ('magnitude', 'depth', 'Magnitude vs Depth'),
        'mag_sig': ('magnitude', 'sig', 'Magnitude vs Significance'),
        'depth_sig': ('depth', 'sig', 'Depth vs Significance')
    }
    
    x_var, y_var, title = axes_map.get(axes_selection, ('magnitude', 'depth', 'Magnitude vs Depth'))
    
    if x_var not in d.columns or y_var not in d.columns:
        fig = go.Figure()
        fig.add_annotation(text=f"Variables not available", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    size = None
    if 'mag' != x_var and 'mag' != y_var:
        size = 'magnitude'
    elif 'depth' != x_var and 'depth' != y_var:
        size = 'depth'
    elif 'sig' != x_var and 'sig' != y_var:
        size = 'sig'
    
    if size in d.columns:
        d['size_scaled'] = d[size] ** 10 / 10000000
    else:
        d['size_scaled'] = None

    fig = px.scatter(
        d.dropna(subset=[x_var, y_var]),
        x=x_var,
        y=y_var,
        size='size_scaled' if size in d.columns else None,
        color='alert' if 'alert' in d.columns else None,
        hover_data=['title'],
        category_orders={'alert': ALERTS} if 'alert' in d.columns else None,
        color_discrete_map=colores_alerta if 'alert' in d.columns else None,
        labels={'alert': 'Alert'}
    )
    fig.update_layout(
        height=350,
        xaxis_title=x_var.capitalize(),
        yaxis_title=y_var.capitalize()
    )
    return fig

@app.callback(
    Output('venn-diagram', 'src'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_venn(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    tsunami_set = set(d[d['tsunami']==1].index)
    red_alert_set = set(d[d['alert']==traduccion_colores['red']].index) if 'alert' in d.columns else set()
    mag6_set = set(d[d['magnitude']>6].index)
    fig, ax = plt.subplots(figsize=(4,4))
    
    sets = [red_alert_set, tsunami_set, mag6_set]
    labels = ['Red Alert', 'Tsunami', 'Magnitude>6']
    
    # Filter empty sets
    non_empty = [(s, l) for s, l in zip(sets, labels) if len(s) > 0]
    
    if len(non_empty) == 0:
        # All sets are empty
        ax.text(0.5, 0.5, 'No data to display', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
    elif len(non_empty) == 1:
        # Only one non-empty set: show set size
        ax.text(0.5, 0.5, f'{non_empty[0][1]}: {len(non_empty[0][0])} earthquakes', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
    elif len(non_empty) == 2:
        # Two non-empty sets
        s1, s2 = non_empty[0][0], non_empty[1][0]
        v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                  set_labels=(non_empty[0][1], non_empty[1][1]), ax=ax)
        # Hide subset labels with value 0
        for text in v.subset_labels:
            if text is not None and text.get_text() == '0':
                text.set_visible(False)
    else:
        # Three non-empty sets: check for duplicates
        if red_alert_set == tsunami_set == mag6_set:
            # All sets are identical
            ax.text(0.5, 0.5, f'All sets are identical\n{len(red_alert_set)} earthquakes', 
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis('off')
        elif red_alert_set == tsunami_set or red_alert_set == mag6_set or tsunami_set == mag6_set:
            # Two sets are identical
            if red_alert_set == tsunami_set:
                s1, s2 = red_alert_set, mag6_set
                v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                          set_labels=('Red Alert & Tsunami', 'Magnitude>6'), ax=ax)
            elif red_alert_set == mag6_set:
                s1, s2 = red_alert_set, tsunami_set
                v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                          set_labels=('Red Alert & Magnitude>6', 'Tsunami'), ax=ax)
            else:  # tsunami_set == mag6_set
                s1, s2 = red_alert_set, tsunami_set
                v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                          set_labels=('Red Alert', 'Tsunami & Magnitude>6'), ax=ax)
            # Hide subset labels with value 0
            for text in v.subset_labels:
                if text is not None and text.get_text() == '0':
                    text.set_visible(False)
        else:
            # All sets are different: calculate subset sizes explicitly
            only_red = len(red_alert_set - tsunami_set - mag6_set)
            only_tsunami = len(tsunami_set - red_alert_set - mag6_set)
            red_and_tsunami = len((red_alert_set & tsunami_set) - mag6_set)
            only_mag6 = len(mag6_set - red_alert_set - tsunami_set)
            red_and_mag6 = len((red_alert_set & mag6_set) - tsunami_set)
            tsunami_and_mag6 = len((tsunami_set & mag6_set) - red_alert_set)
            all_three = len(red_alert_set & tsunami_set & mag6_set)
            
            v = venn3(subsets=(only_red, only_tsunami, red_and_tsunami, only_mag6, 
                              red_and_mag6, tsunami_and_mag6, all_three),
                      set_labels=('Red Alert', 'Tsunami', 'Magnitude>6'), ax=ax)
            # Hide subset labels with value 0
            for text in v.subset_labels:
                if text is not None and text.get_text() == '0':
                    text.set_visible(False)
    
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', transparent=True)
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return f'data:image/png;base64,{encoded}'

@app.callback(
    Output('radar-graph', 'figure'),
    Input('radar-select', 'value')
)
def update_radar(selected_idx):
    if selected_idx is None or selected_idx >= len(df_top10):
        return radar_placeholder()
    eq = df_top10.iloc[selected_idx]
    variables = ['magnitude', 'depth', 'sig', 'cdi', 'mmi']
    labels = ['Magnitude','Depth','Significance','CDI','MMI']
    values = []
    for v in variables:
        if v in df.columns and pd.notna(eq.get(v)):
            vmin, vmax = df[v].min(), df[v].max()
            if pd.isna(vmin) or pd.isna(vmax) or vmax==vmin:
                values.append(50)
            else:
                values.append((eq[v]-vmin)/(vmax-vmin)*100)
        else:
            values.append(0)
    values.append(values[0])
    labels_closed = labels + [labels[0]]
    
    fig = go.Figure(go.Scatterpolar(
        r=values,
        theta=labels_closed,
        fill='toself',
        name=eq.get('title','event'),
        hovertemplate='%{theta}: %{r:.2f}%'
    ))
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                range=[0,100],
                showticklabels=False
            )
        ), 
        height=350
    )
    return fig

@app.callback(
    Output('density-graph', 'figure'),
    Input('density-variable', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_density(variable, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):

    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    x = d[variable].dropna()
    if len(x) == 0:
        fig = go.Figure()
        fig.add_annotation(text=f"No data for {variable}", x=0.5, y=0.5, showarrow=False)
        return fig

    hist = go.Histogram(
        x=x,
        histnorm='probability density',
        nbinsx=40,
        name='Histogram',
        opacity=0.5,
        marker_color='lightblue'
    )

    kde = gaussian_kde(x)
    xs = np.linspace(x.min(), x.max(), 300)
    ys = kde(xs)
    line = go.Scatter(
        x=xs,
        y=ys,
        mode='lines',
        line=dict(color='royalblue', width=2.5),
        name='Density (KDE)'
    )

    fig = go.Figure(data=[hist, line])
    fig.update_layout(
        xaxis_title=variable.capitalize(),
        yaxis_title='Density',
        height=350,
        template='plotly_white',
        legend=dict(
            orientation="h",  # Horizontal legend
            yanchor="bottom",
            y=1.02,  # Position above the graph
            xanchor="center",
            x=0.5
        ),
        margin=dict(l=50, r=20, t=40, b=50)  # Tighter margins
    )
    return fig

@app.callback(
    Output('pie-waffle-graph', 'figure'),
    Input('pie-variable', 'value'),
    Input('pie-chart-type', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_pie_waffle(variable, chart_type, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if variable not in d.columns:
        fig = go.Figure()
        fig.add_annotation(text=f"Column '{variable}' not found", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        return fig
    
    d_clean = d[d[variable].notna()].copy()
    
    if len(d_clean) == 0:
        fig = go.Figure()
        fig.add_annotation(text=f"No data available for {variable}", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    if chart_type == 'pie':
        fig = px.pie(
            d_clean,
            names=variable,
            color=variable,
            color_discrete_map=colores_alerta if variable == 'alert' else None,
            category_orders={'alert': ALERTS} if variable == 'alert' else None
        )
        fig.update_layout(height=350)
    elif chart_type == 'waffle':
        counts = d_clean[variable].value_counts().to_dict()
        counts = {k: v for k, v in counts.items() if v >= 10}
        other_count = len(d_clean) - sum(counts.values())
        if other_count > 0:
            counts['other'] = other_count
            
        fig_mpl = plt.figure(
            FigureClass=Waffle,
            rows=10,
            columns=10,
            values=counts,
            labels=[f"{k}: {v}" for k, v in counts.items()],
            colors=[colores_alerta.get(k, None) for k in counts.keys()] if variable == 'alert' else None,
            legend={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
            figsize=(8, 5)
        )
        
        buf = io.BytesIO()
        fig_mpl.savefig(buf, format='png', bbox_inches='tight', dpi=100)
        buf.seek(0)
        encoded = base64.b64encode(buf.read()).decode()
        plt.close(fig_mpl)
        
        fig = go.Figure()
        fig.add_layout_image(
            dict(
                source=f'data:image/png;base64,{encoded}',
                xref="paper", yref="paper",
                x=0, y=1,
                sizex=1, sizey=1,
                xanchor="left", yanchor="top",
                sizing="contain",
                layer="below"
            )
        )
        fig.update_xaxes(visible=False, range=[0, 1])
        fig.update_yaxes(visible=False, range=[0, 1])
        fig.update_layout(
            height=350,
            margin=dict(l=0, r=0, t=0, b=0),
            plot_bgcolor='white'
        )
    else:
        fig = go.Figure()
        fig.add_annotation(text="Chart type not supported", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)

    return fig

def calendar_heatmap(ts_df):
    dfc = ts_df.copy()
    dfc['year_month'] = dfc['date_time'].dt.to_period('ME')
    counts = dfc.groupby('year_month', observed=True).size().reset_index(name='count')
    if counts.empty:
        return go.Figure()
    counts['year'] = counts['year_month'].dt.year
    counts['month'] = counts['year_month'].dt.month
    pivot = counts.pivot(index='year', columns='month', values='count').fillna(0)
    fig = go.Figure(data=go.Heatmap(z=pivot.values, x=list(pivot.columns), y=list(pivot.index), colorscale='Blues'))
    fig.update_layout(title='Earthquakes per month (heatmap)', xaxis_title='Month', yaxis_title='Year', height=400)
    return fig

@app.callback(
    Output('time-line-graph', 'figure'),
    Input('time-line-period', 'value'),
    Input('time-line-smoothing', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_time_line(value, window, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    ts = d.dropna(subset=['date_time']).copy()
    
    if len(ts) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        return fig
    
    # Determine aggregation period
    if value == '':
        # No aggregation: show all individual earthquakes
        mag_time = ts[['date_time', 'magnitude']].sort_values('date_time')
        x_label = 'Date'
        y_label = 'Magnitude'
        x_data = mag_time['date_time']
        y_data = mag_time['magnitude']
    elif value in ['H', 'D', 'ME', 'YE']:
        if value == 'H':
            ts['x'] = ts['date_time'].dt.hour
            rango_tiempo = pd.DataFrame({'x': range(24)})
            x_label = 'Hour of Day'
        elif value == 'D':
            ts['x'] = ts['date_time'].dt.day
            rango_tiempo = pd.DataFrame({'x': range(1, 32)})
            x_label = 'Day of Month'
        elif value == 'ME':
            ts['x'] = ts['date_time'].dt.month
            rango_tiempo = pd.DataFrame({'x': range(1, 13)})
            x_label = 'Month of Year'
        elif value == 'YE':
            ts['x'] = ts['date_time'].dt.year
            rango_tiempo = pd.DataFrame({'x': range(ts['date_time'].dt.year.min(), ts['date_time'].dt.year.max() + 1)})
            x_label = 'Year'
        y_label = 'Number of Earthquakes'
        
        mag_time = ts.groupby('x', observed=True).size().reset_index(name='count')
        mag_time = rango_tiempo.merge(mag_time, on='x', how='left').fillna(0)
        mag_time['count'] = mag_time['count'].astype(int)
        
        x_data = mag_time['x']
        y_data = mag_time['count']
    else:
        fig = go.Figure()
        fig.add_annotation(text="Period not supported", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        return fig
    
    y_min = math.floor(y_data.min()) if len(y_data) > 0 else 0
    y_max = y_data.max() if len(y_data) > 0 else 1
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=x_data,
        y=y_data,
        mode='markers',
        marker=dict(size=10, color='royalblue'),
        name='Magnitude' if value == '' else 'Number of Earthquakes',
        hovertemplate='%{x}<br>' + y_label + ': %{y:.2f}<extra></extra>'
    ))
    
    # Add smoothed line only if there is enough aggregated data
    if window is not None and len(y_data) > window and window > 1:
        # use simple moving average
        y_smooth = y_data.rolling(window=window, center=True).mean()
        
        fig.add_trace(go.Scatter(
            x=x_data,
            y=y_smooth,
            mode='lines',
            line=dict(color='darkblue', width=3, dash='solid'),
            name='Smoothed Trend',
            hovertemplate='%{x}<br>Smoothed: %{y:.2f}<extra></extra>'
        ))
    
    fig.update_layout(
        xaxis_title=x_label,
        yaxis_title=y_label,
        yaxis=dict(range=[y_min, y_max * 1.05]),
        height=350,
        template='plotly_white',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )
    return fig

@app.callback(
    Output('time-animation-graph', 'figure'),
    Input('time-animation-dimension', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_animation(dim, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    ts = d.dropna(subset=['date_time', 'latitude', 'longitude']).copy()
    
    if len(ts) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        return fig
    
    ts['year'] = ts['date_time'].dt.year
    
    first_year = ts['year'].min()
    sample_df = pd.DataFrame({
        'date_time': [pd.Timestamp(f'{first_year}-01-01')] * len(ALERTS),
        'latitude': [None] * len(ALERTS),
        'longitude': [None] * len(ALERTS),
        'magnitude': [0] * len(ALERTS),
        'alert': ALERTS,
        'title': ['Sample Alert'] * len(ALERTS),
        'year': [first_year] * len(ALERTS)
    })
    ts = pd.concat([ts, sample_df], ignore_index=True)
    ts['magnitude_scaled'] = ts['magnitude'] ** 10 / 10000000
    
    ts = ts.sort_values('year', ascending=True)
    
    fig = px.scatter_geo(
        ts,
        lat='latitude',
        lon='longitude',
        size='magnitude_scaled',
        color='alert',
        color_discrete_map=colores_alerta,
        category_orders={'alert': ALERTS},
        hover_name='title',
        hover_data={'magnitude': True, 'magnitude_scaled': False, 'alert': True, 
                    'latitude': ':.2f', 'longitude': ':.2f'},
        animation_frame='year',
        projection='orthographic' if dim == 3 else 'natural earth'
    )
    
    fig.update_layout(
        height=350,
        showlegend=False
    )
    
    return fig

@app.callback(
    Output('alert-timeseries-graph', 'figure'),
    Input('alert-timeseries-period', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_alert_timeseries(period, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    ts = d.dropna(subset=['date_time']).copy()
    
    if len(ts) == 0 or 'alert' not in ts.columns:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        return fig
    
    # Group according to selected period
    if period == 'H':
        # Group by hour of day (independent of day, month and year)
        ts['period'] = ts['date_time'].dt.hour
        alert_counts = ts.groupby(['period', 'alert'], observed=True).size().reset_index(name='count')
        
        # Ensure all hours are present
        all_hours = list(range(24))
        pivot = alert_counts.pivot(index='period', columns='alert', values='count').fillna(0)
        pivot = pivot.reindex(all_hours, fill_value=0)
        
        x_values = list(pivot.index)
        x_title = 'Hour of Day'
    elif period == 'M':
        # Group by month (independent of year)
        ts['period'] = ts['date_time'].dt.month
        alert_counts = ts.groupby(['period', 'alert'], observed=True).size().reset_index(name='count')
        
        # Create month names for x-axis
        month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 
                      5: 'May', 6: 'June', 7: 'July', 8: 'August',
                      9: 'September', 10: 'October', 11: 'November', 12: 'December'}
        alert_counts['period_label'] = alert_counts['period'].map(month_names)
        
        # Ensure all months are present
        all_months = list(range(1, 13))
        pivot = alert_counts.pivot(index='period', columns='alert', values='count').fillna(0)
        pivot = pivot.reindex(all_months, fill_value=0)
        
        x_values = [month_names[m] for m in pivot.index]
        x_title = 'Month of Year'
        
    elif period == 'D':
        # Group by day of month (independent of month and year)
        ts['period'] = ts['date_time'].dt.day
        alert_counts = ts.groupby(['period', 'alert'], observed=True).size().reset_index(name='count')
        
        # Ensure all days (1-31) are present
        all_days = list(range(1, 32))
        pivot = alert_counts.pivot(index='period', columns='alert', values='count').fillna(0)
        pivot = pivot.reindex(all_days, fill_value=0)
        
        x_values = list(pivot.index)
        x_title = 'Day of Month'

    elif period == 'Y':
        # Group by year
        ts['period'] = ts['date_time'].dt.year
        alert_counts = ts.groupby(['period', 'alert'], observed=True).size().reset_index(name='count')
        
        # Create pivot for stacked area
        pivot = alert_counts.pivot(index='period', columns='alert', values='count').fillna(0)
        
        x_values = list(pivot.index)
        x_title = 'Year'
    else:
        fig = go.Figure()
        fig.add_annotation(text="Period not supported", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        return fig
    
    # Create figure with stacked area
    fig = go.Figure()
    
    for alert in ALERTS:
        if alert in pivot.columns:
            fig.add_trace(go.Scatter(
                x=x_values,
                y=pivot[alert],
                mode='lines',
                name=alert,
                stackgroup='one',
                line=dict(color=colores_alerta.get(alert), width=0),
                fillcolor=colores_alerta.get(alert),
                hovertemplate=f'{alert}: %{{y}}<extra></extra>'
            ))
    
    fig.update_layout(
        xaxis_title=x_title,
        yaxis_title='Number of Earthquakes',
        height=350,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )
    
    # If by month or day, adjust x-axis to show all values
    if period in ['M', 'D']:
        fig.update_xaxes(type='category')
    
    return fig

@app.callback(
    Output('calendar-heatmap-graph', 'figure'),
    Input('calendar-granularity', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_calendar_heatmap(granularity, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    ts = d.dropna(subset=['date_time']).copy()
    
    if len(ts) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        return fig
    
    ts['year'] = ts['date_time'].dt.year
    
    if granularity == 'hour':
        ts['hour'] = ts['date_time'].dt.hour
        pivot = ts.groupby(['year', 'hour'], observed=True).size().reset_index(name='count')
        pivot_table = pivot.pivot(index='year', columns='hour', values='count').fillna(0)
        
        fig = go.Figure(data=go.Heatmap(
            z=pivot_table.values,
            x=list(pivot_table.columns),
            y=list(pivot_table.index),
            colorscale='YlOrRd',
            hovertemplate='Year: %{y}<br>Hour: %{x}:00<br>Earthquakes: %{z}<extra></extra>'
        ))
        fig.update_layout(xaxis_title='Hour of Day', yaxis_title='Year',
        xaxis=dict(
            tickmode='linear',
            tick0=0,
            dtick=1,
            tickangle=45
        ))
    elif granularity == 'day':
        ts['day_of_month'] = ts['date_time'].dt.day
        pivot = ts.groupby(['year', 'day_of_month'], observed=True).size().reset_index(name='count')
        pivot_table = pivot.pivot(index='year', columns='day_of_month', values='count').fillna(0)
        fig = go.Figure(data=go.Heatmap(
            z=pivot_table.values,
            x=list(pivot_table.columns),
            y=list(pivot_table.index),
            colorscale='YlOrRd',
            hovertemplate='Year: %{y}<br>Day of Month: %{x}<br>Earthquakes: %{z}<extra></extra>'
        ))
        fig.update_layout(xaxis_title='Day of Month', yaxis_title='Year',
        xaxis=dict(
            tickmode='linear',
            tick0=1,
            dtick=1,
            tickangle=60
        ))
    elif granularity == 'month':
        ts['month_of_year'] = ts['date_time'].dt.month
        month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 
                      5: 'May', 6: 'June', 7: 'July', 8: 'August',
                      9: 'September', 10: 'October', 11: 'November', 12: 'December'}
        pivot = ts.groupby(['year', 'month_of_year'], observed=True).size().reset_index(name='count')
        pivot_table = pivot.pivot(index='year', columns='month_of_year', values='count').fillna(0)
        fig = go.Figure(data=go.Heatmap(
            z=pivot_table.values,
            x=list(pivot_table.columns.map(month_names)),
            y=list(pivot_table.index),
            colorscale='YlOrRd',
            hovertemplate='Year: %{y}<br>Month: %{x}<br>Earthquakes: %{z}<extra></extra>'
        ))
        fig.update_layout(xaxis_title='Month of Year', yaxis_title='Year',
        xaxis=dict(
            tickmode='linear',
            tick0=0,
            dtick=1
        ))
    else:
        fig = go.Figure()
        fig.add_annotation(text="Granularity not supported", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        return fig

    fig.update_layout(height=350, template='plotly_white')
    return fig


@app.callback(
    Output('polar-chart-graph', 'figure'),
    Input('polar-type', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_polar_chart(chart_type, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    ts = d.dropna(subset=['date_time']).copy()
    
    if len(ts) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        return fig
    
    if chart_type == 'hour':
        ts['hour'] = ts['date_time'].dt.hour
        counts = ts['hour'].value_counts().sort_index()
        labels = [f'{h}:00' for h in range(24)]
        values = [counts.get(h, 0) for h in range(24)]
        theta = labels
    elif chart_type == 'month':
        ts['month'] = ts['date_time'].dt.month
        counts = ts['month'].value_counts().sort_index()
        month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 
                      5: 'May', 6: 'June', 7: 'July', 8: 'August', 9: 'September',
                      10: 'October', 11: 'November', 12: 'December'}
        values = [counts.get(m, 0) for m in range(1, 13)]
        theta = [month_names[m] for m in range(1, 13)]
    elif chart_type == 'day':
        ts['day'] = ts['date_time'].dt.day
        counts = ts['day'].value_counts().sort_index()
        labels = [str(d) for d in range(1, 32)]
        values = [counts.get(d, 0) for d in range(1, 32)]
        theta = labels
    else:
        fig = go.Figure()
        fig.add_annotation(text="Type not supported", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        return fig
    
    fig = go.Figure(go.Barpolar(
        r=values,
        theta=theta,
        marker_color='royalblue',
        marker_line_color='white',
        marker_line_width=2,
        opacity=0.8
    ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(showticklabels=False, ticks=''),
            angularaxis=dict(direction='clockwise')
        ),
        height=350,
        template='plotly_white'
    )
    return fig

@app.callback(
    Output('espacio-bar-graph', 'figure'),
    Input('espacio-bar-variable', 'value'),
    Input('espacio-bar-type', 'value'),
    Input('espacio-bar-desconocido', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_bar(variable, chart_type, incluir_nulos, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if not incluir_nulos or True not in incluir_nulos:
        d = d[d[variable] != VALOR_SUSTITUTO_NULO]
    
    if chart_type == 'bar':
        top_counts = d[variable].value_counts().nlargest(12).index
        d = d[d[variable].isin(top_counts)]
        counts = d[variable].value_counts()
        
        fig = px.bar(
            x=counts.values,
            y=counts.index,
            orientation='h',
            color=counts.index if variable == 'alert' else None,
            color_discrete_map=colores_alerta if variable == 'alert' else None
        )
        fig.update_layout(
            xaxis_title='Number of Earthquakes',
            yaxis_title=variable.capitalize(),
            showlegend=False,
            height=350,
            uirevision='bar'
        )
        return fig
    elif chart_type == 'wordcloud':
        text_list = d[variable].dropna().astype(str).tolist()
        
        if len(text_list) == 0:
            fig = go.Figure()
            fig.add_annotation(text="Not enough text", x=0.5, y=0.5, showarrow=False)
            fig.update_layout(height=350)
            return fig
        
        # Count frequencies
        freq_dict = Counter(text_list)
        
        # If there are very few unique words, show message
        if len(freq_dict) == 0:
            fig = go.Figure()
            fig.add_annotation(text="Not enough text", x=0.5, y=0.5, showarrow=False)
            fig.update_layout(height=350)
            return fig
        
        wc = WordCloud(
            width=1200,
            height=600,
            background_color='white',
            relative_scaling=0.5,
            min_font_size=10,
            max_font_size=150,
            margin=20,
            prefer_horizontal=0.7,
            collocations=False,
            regexp=r"\w[\w ']+"  # Allows spaces within words
        ).generate_from_frequencies(freq_dict) 
        
        wc_image = wc.to_image()
        
        img_array = np.array(wc_image)
        mask = img_array.sum(axis=2) < 255 * 3
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        
        if rows.any() and cols.any():
            ymin, ymax = np.where(rows)[0][[0, -1]]
            xmin, xmax = np.where(cols)[0][[0, -1]]
            margin = 20
            ymin = max(0, ymin - margin)
            ymax = min(img_array.shape[0], ymax + margin)
            xmin = max(0, xmin - margin)
            xmax = min(img_array.shape[1], xmax + margin)
            
            wc_image = wc_image.crop((xmin, ymin, xmax, ymax))
        
        buf = io.BytesIO()
        wc_image.save(buf, format='PNG')
        buf.seek(0)
        encoded = base64.b64encode(buf.read()).decode()
        
        fig = go.Figure()
        
        fig.add_layout_image(
            x=0.5,
            y=0.5,
            source=f'data:image/png;base64,{encoded}',
            xref="paper",
            yref="paper",
            sizex=1,
            sizey=1,
            xanchor="center",
            yanchor="middle",
            sizing="contain",
            layer="below"
        )
        
        fig.update_xaxes(
            visible=False,
            range=[0, 1],
            fixedrange=True,
            showgrid=False
        )
        
        fig.update_yaxes(
            visible=False,
            range=[0, 1],
            fixedrange=True,
            showgrid=False
        )
        
        fig.update_layout(
            height=350,
            margin=dict(l=0, r=0, t=0, b=0),
            plot_bgcolor='white',
            paper_bgcolor='white',
            xaxis_showgrid=False,
            yaxis_showgrid=False,
            uirevision=None,
            template=None
        )
        
        return fig
    else:
        fig = go.Figure()
        fig.add_annotation(text="Chart type not supported", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
@app.callback(
    Output('espacio-treemap-graph', 'figure'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_treemap(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    required_cols = ['continent', 'country', 'location']
    if not all(col in d.columns for col in required_cols) or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    d_clean = d[required_cols].fillna(VALOR_SUSTITUTO_NULO)
    d_clean['count'] = 1
    
    fig = px.treemap(
        d_clean,
        path=['continent', 'country', 'location'],
        values='count',
        color='continent'
    )
    fig.update_layout(height=350, margin=dict(l=0, r=0, t=30, b=0))
    return fig

@app.callback(
    Output('espacio-latitude-area-graph', 'figure'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_latitude_area(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if 'latitude' not in d.columns or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    # Create latitude bins
    lat_bins = np.linspace(-90, 90, 50)
    d['lat_bin'] = pd.cut(d['latitude'], bins=lat_bins)
    lat_counts = d.groupby('lat_bin', observed=True).size().reset_index(name='count')
    lat_counts['lat_center'] = lat_counts['lat_bin'].apply(lambda x: x.mid)
    
    fig = go.Figure()
    
    # Split into northern and southern hemisphere
    norte = lat_counts[lat_counts['lat_center'] >= 0]
    sur = lat_counts[lat_counts['lat_center'] < 0]
    
    fig.add_trace(go.Scatter(
        x=norte['count'], y=norte['lat_center'],
        fill='tozerox', fillcolor='rgba(255, 100, 100, 0.5)',
        line=dict(color='red'), name='Northern Hemisphere'
    ))
    
    fig.add_trace(go.Scatter(
        x=sur['count'], y=sur['lat_center'],
        fill='tozerox', fillcolor='rgba(100, 100, 255, 0.5)',
        line=dict(color='blue'), name='Southern Hemisphere'
    ))
    
    # Equator line
    fig.add_hline(y=0, line_dash="dash", line_color="black", 
                  annotation_text="Equator", annotation_position="right")
    
    fig.update_layout(
        xaxis_title='Number of Earthquakes',
        yaxis_title='Latitude (°)',
        height=350,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )
    return fig

@app.callback(
    Output('espacio-folium-map', 'srcDoc'),
    Input('espacio-folium-basemap', 'value'),
    Input('espacio-folium-layers', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_folium_map(basemap, layers, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if 'latitude' not in d.columns or 'longitude' not in d.columns or len(d) == 0:
        # Empty map
        m = folium.Map(location=[0, 0], zoom_start=2, tiles=basemap,
            no_wrap=True,
            world_copy_jump=False
        )
        folium.Marker([0, 0], popup="No data available").add_to(m)
        return m._repr_html_()
    
    d_clean = d.dropna(subset=['latitude', 'longitude', 'magnitude', 'alert']).copy()
    
    if len(d_clean) == 0:
        m = folium.Map(location=[0, 0], zoom_start=2, tiles=basemap,
            no_wrap=True,
            world_copy_jump=False
        )
        return m._repr_html_()
    
    # Create map centered on average locations
    center_lat = d_clean['latitude'].mean()
    center_lon = d_clean['longitude'].mean()
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=2,
        tiles=basemap,
        control_scale=True,
        no_wrap=True,
        world_copy_jump=False
    )
    
    # Heatmap Layer
    if layers and 'heatmap' in layers:
        heat_data = [[row['latitude'], row['longitude'], row['magnitude']] 
                     for idx, row in d_clean.iterrows()]
        
        HeatMap(
            heat_data,
            name='Heatmap',
            min_opacity=0.3,
            max_opacity=0.8,
            radius=15,
            blur=20,
            gradient={
                0.0: 'blue',
                0.3: 'lime',
                0.5: 'yellow',
                0.7: 'orange',
                1.0: 'red'
            }
        ).add_to(m)
    
    # Clustered Markers Layer
    if layers and 'clusters' in layers:
        marker_cluster = MarkerCluster(name='Earthquake Clusters').add_to(m)
        
        for idx, row in d_clean.iterrows():
            # Determine color according to alert
            alert_color = {
                'green': 'green',
                'yellow': 'orange',
                'orange': 'orange',
                'red': 'red'
            }.get(row['alert'], 'gray')
            
            # Icon according to tsunami
            icon_shape = 'circle' if row.get('tsunami', 0) == 1 else 'info-sign'
            
            popup_html = f"""
            <div style='width: 200px'>
                <h4>{row.get('title', 'Earthquake')}</h4>
                <p><b>Magnitude:</b> {row['magnitude']:.2f}</p>
                <p><b>Depth:</b> {row.get('depth', 'N/A')} km</p>
                <p><b>Alert:</b> {row['alert']}</p>
                <p><b>Tsunami:</b> {'Yes' if row.get('tsunami', 0) == 1 else 'No'}</p>
                <p><b>Date:</b> {row.get('date_time', 'N/A')}</p>
            </div>
            """
            
            folium.Marker(
                location=[row['latitude'], row['longitude']],
                popup=folium.Popup(popup_html, max_width=250),
                icon=folium.Icon(color=alert_color, icon=icon_shape),
                tooltip=f"Mag: {row['magnitude']:.1f}"
            ).add_to(marker_cluster)
    
    # Circles by Magnitude Layer
    if layers and 'circles' in layers:
        # Create colormap for alerts
        feature_group = folium.FeatureGroup(name='Circles by Magnitude')
        
        for idx, row in d_clean.iterrows():
            alert_color_map = {
                'green': '#00ff00',
                'yellow': '#ffff00',
                'orange': '#ff8800',
                'red': '#ff0000'
            }
            color = alert_color_map.get(row['alert'], '#808080')
            
            # Radius proportional to magnitude
            radius = row['magnitude'] ** 8 / 100
            
            popup_html = f"""
            <b>{row.get('title', 'Earthquake')}</b><br>
            Mag: {row['magnitude']:.2f}<br>
            Alert: {row['alert']}<br>
            Tsunami: {'Yes' if row.get('tsunami', 0) == 1 else 'No'}
            """
            
            # Solid circle for normal, thick border for tsunamis
            if row.get('tsunami', 0) == 1:
                folium.Circle(
                    location=[row['latitude'], row['longitude']],
                    radius=radius,
                    popup=popup_html,
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.3,
                    weight=4,
                    opacity=0.8
                ).add_to(feature_group)
            else:
                folium.Circle(
                    location=[row['latitude'], row['longitude']],
                    radius=radius,
                    popup=popup_html,
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.5,
                    weight=1,
                    opacity=0.7
                ).add_to(feature_group)
        
        feature_group.add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # custom legend
    legend_html = '''
    <div style="position: fixed; 
                bottom: 40px; right: 40px; width: 140px; height: auto; 
                background-color: white; z-index:9999; font-size:14px;
                border:2px solid grey; border-radius: 5px; padding: 5px">
        <p style="margin: 0 0 3px 0;"><b>Alerts</b></p>
        <p style="margin: 1px;"><span style="color: red;">●</span> Red</p>
        <p style="margin: 1px;"><span style="color: orange;">●</span> Orange</p>
        <p style="margin: 1px;"><span style="color: #ffff00;">●</span> Yellow</p>
        <p style="margin: 1px;"><span style="color: green;">●</span> Green</p>
        <hr style="margin: 3px 0;">
        <p style="margin: 0 0 3px 0;"><b>Circle Types</b></p>
        <p style="margin: 1px;"><b>○</b> Tsunami</p>
        <p style="margin: 1px;"><b>●</b> Earthquake Only</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m._repr_html_()

@app.callback(
    Output('espacio-choropleth-graph', 'figure'),
    Input('espacio-choropleth-metric', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_choropleth(metric, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if 'country' not in d.columns or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=300)
        return fig
    
    # Mapping of countries to ISO codes
    country_iso = {
        'Afghanistan': 'AFG',
        'Algeria': 'DZA',
        'Antarctica': 'ATA',
        'Argentina': 'ARG',
        'Azerbaijan': 'AZE',
        'Bolivia': 'BOL',
        'Botswana': 'BWA',
        'Brazil': 'BRA',
        'Canada': 'CAN',
        'Chile': 'CHL',
        'Colombia': 'COL',
        'Costa Rica': 'CRI',
        'Ecuador': 'ECU',
        'El Salvador': 'SLV',
        'Fiji': 'FJI',
        'Greece': 'GRC',
        'Guatemala': 'GTM',
        'Haiti': 'HTI',
        'Iceland': 'ISL',
        'India': 'IND',
        'Indonesia': 'IDN',
        'Iran': 'IRN',
        'Italy': 'ITA',
        'Japan': 'JPN',
        'Kyrgyzstan': 'KGZ',
        'Martinique': 'MTQ',
        'Mexico': 'MEX',
        'Mongolia': 'MNG',
        'Mozambique': 'MOZ',
        'Myanmar': 'MMR',
        'Nepal': 'NPL',
        'New Zealand': 'NZL',
        'Nicaragua': 'NIC',
        'Pakistan': 'PAK',
        'Panama': 'PAN',
        'Papua New Guinea': 'PNG',
        "People's Republic of China": 'CHN',
        'Peru': 'PER',
        'Philippines': 'PHL',
        'Russia': 'RUS',
        'Russian Federation (the)': 'RUS',
        'Saudi Arabia': 'SAU',
        'Solomon Islands': 'SLB',
        'South Georgia and the South Sandwich Islands': 'SGS',
        'Taiwan': 'TWN',
        'Tajikistan': 'TJK',
        'Tanzania': 'TZA',
        'Tonga': 'TON',
        'Trinidad and Tobago': 'TTO',
        'Turkey': 'TUR',
        'Turkiye': 'TUR',
        'Turkmenistan': 'TKM',
        'United Kingdom of Great Britain and Northern Ireland (the)': 'GBR',
        'United States of America': 'USA',
        'Vanuatu': 'VUT',
        'Venezuela': 'VEN'
    }
    
    # Calculate metric
    if metric == 'count':
        agg = d.groupby('country', observed=True).size().reset_index(name='value')
        colorbar_title = 'Number'
    elif metric == 'mean_mag':
        agg = d.groupby('country', observed=True)['magnitude'].mean().reset_index(name='value')
        colorbar_title = 'Avg. Mag.'
    elif metric == 'max_mag':
        agg = d.groupby('country', observed=True)['magnitude'].max().reset_index(name='value')
        colorbar_title = 'Max. Mag.'
    elif metric == 'mean_sig':
        agg = d.groupby('country', observed=True)['sig'].mean().reset_index(name='value')
        colorbar_title = 'Avg. Sig.'
    elif metric == 'tsunami_count':
        agg = d.groupby('country', observed=True)['tsunami'].sum().reset_index(name='value')
        colorbar_title = 'Tsunamis'
    elif metric == 'mean_alert':
        alert_values = {'green': 1, 'yellow': 2, 'orange': 3, 'red': 4}
        d['alert_numeric'] = d['alert'].map(alert_values)
        agg = d.groupby('country', observed=True)['alert_numeric'].mean().reset_index(name='value')
        colorbar_title = 'Avg. Alert'
    elif metric == 'mean_mmi':
        agg = d.groupby('country', observed=True)['mmi'].mean().reset_index(name='value')
        colorbar_title = 'Avg. MMI'
    else:
        agg = d.groupby('country', observed=True).size().reset_index(name='value')
        colorbar_title = 'Number'
    
    agg['iso_alpha'] = agg['country'].map(country_iso)
    agg = agg.dropna(subset=['iso_alpha'])
    
    fig = px.choropleth(
        agg,
        locations='iso_alpha',
        color='value',
        hover_name='country',
        color_continuous_scale='Reds'
    )
    fig.update_layout(
        height=300,
        margin=dict(l=0, r=0, t=0, b=0),
        coloraxis_colorbar=dict(title=colorbar_title)
    )
    return fig

@app.callback(
    Output('espacio-3d-globe-graph', 'figure'),
    Input('espacio-3d-color', 'value'),
    Input('espacio-3d-factor', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_3d_globe(color_by, factor_multiplicativo, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    required_cols = ['latitude', 'longitude', 'depth', 'magnitude']
    if not all(col in d.columns for col in required_cols) or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=500)
        return fig
    
    d_clean = d.dropna(subset=required_cols).head(200)
    
    # Convert lat/lon to 3D coordinates
    R = 1.0
    lat_rad = np.radians(d_clean['latitude'])
    lon_rad = np.radians(d_clean['longitude'])
    
    # Surface
    x_surf = R * np.cos(lat_rad) * np.cos(lon_rad)
    y_surf = R * np.cos(lat_rad) * np.sin(lon_rad)
    z_surf = R * np.sin(lat_rad)
    
    # Depth (towards the center)
    if not factor_multiplicativo or factor_multiplicativo <= 0:
        factor_multiplicativo = 1.0
    earth_radius_km = 6371.0 # Average Earth radius in km
    max_depth_km = d_clean['depth'].max()
    depth_scale = max_depth_km / earth_radius_km
    depth_norm = d_clean['depth'] / max_depth_km
    r_depth = R - depth_norm * depth_scale * factor_multiplicativo
    
    x_depth = r_depth * np.cos(lat_rad) * np.cos(lon_rad)
    y_depth = r_depth * np.cos(lat_rad) * np.sin(lon_rad)
    z_depth = r_depth * np.sin(lat_rad)
    
    fig = go.Figure()

    # Create sphere using lat/lon parameterization
    lon_sphere = np.linspace(-np.pi, np.pi, 100)  # -180 to 180 degrees
    lat_sphere = np.linspace(-np.pi/2, np.pi/2, 100)  # -90 to 90 degrees

    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(lon_sphere, lat_sphere)

    # Convert to 3D coordinates (same formula as your earthquake data)
    x_sphere = R * np.cos(lat_grid) * np.cos(lon_grid)
    y_sphere = R * np.cos(lat_grid) * np.sin(lon_grid)
    z_sphere = R * np.sin(lat_grid)

    # Load texture
    earth_img = load_earth_texture_rgb('2k_earth_specular_map.tif', resolution=(100, 100))

    # Texture image typically has:
    # - Width = longitude (0 to 360 degrees, left to right)
    # - Height = latitude (90 to -90 degrees, top to bottom)
    # Therefore, we may need to flip it
    earth_img_flipped = np.flipud(earth_img)  # Flip vertically for correct latitude orientation
    
    colorscale, surfacecolor = create_colorscale_from_image(earth_img_flipped)

    fig.add_trace(go.Surface(
        x=x_sphere, 
        y=y_sphere, 
        z=z_sphere,
        surfacecolor=surfacecolor,
        colorscale=colorscale,
        showscale=False,
        opacity=0.2,
        hoverinfo='skip',
        lighting=dict(
            ambient=0.5,
            diffuse=0.0,
            specular=0.0,
            fresnel=2
        ),
        hidesurface=False
    ))
    
    # Depth lines
    colors = []
    for idx, row in d_clean.iterrows():
        if color_by == 'alert':
            color = colores_alerta.get(row['alert'], 'gray')
        else:
            color = f"rgb({int(255 * row['depth'] / d_clean['depth'].max())}, 100, 100)"
        colors.append(color)
    
    for i in range(len(d_clean)):
        fig.add_trace(go.Scatter3d(
            x=[x_surf.iloc[i], x_depth.iloc[i]],
            y=[y_surf.iloc[i], y_depth.iloc[i]],
            z=[z_surf.iloc[i], z_depth.iloc[i]],
            mode='lines',
            line=dict(
                color=colors[i],
                width=d_clean['magnitude'].iloc[i] ** 8 / 1000000 #** 4 / 500
            ),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data'
        ),
        height=500,
        margin=dict(l=0, r=0, t=0, b=0)
    )
    return fig

@app.callback(
    Output('espacio-boxplot-graph', 'figure'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_boxplot(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if 'continent' not in d.columns or 'magnitude' not in d.columns or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text="No data available", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    fig = px.box(
        d,
        x='continent',
        y='magnitude',
        color='continent',
        points='outliers'
    )
    fig.update_layout(
        xaxis_title='Continent',
        yaxis_title='Magnitude',
        height=350,
        showlegend=False
    )
    return fig

@app.callback(
    [Output('espacio-prediction-results', 'children'),
     Output('espacio-prediction-map', 'figure')],
    Input('espacio-predict-button', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_espacio_prediction(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    if n_clicks is None or n_clicks == 0:
        return html.Div("Click the button to train the model"), go.Figure()
    
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    required_cols = ['latitude', 'longitude', 'depth', 'magnitude', 'date_time', 'country', 'continent']
    if not all(col in d.columns for col in required_cols) or len(d) < 50:
        return html.Div("Insufficient data to train model (minimum 50 events)"), go.Figure()
    
    # Prepare data
    d_model = d[required_cols].dropna().copy()
    
    # Temporal features
    d_model['year'] = d_model['date_time'].dt.year
    d_model['month'] = d_model['date_time'].dt.month
    d_model['day'] = d_model['date_time'].dt.day
    d_model['hour'] = d_model['date_time'].dt.hour
    d_model['timestamp'] = d_model['date_time'].astype('int64') / 10**9
    
    # Encode categorical variables
    le_country = LabelEncoder()
    le_continent = LabelEncoder()
    
    d_model['country_encoded'] = le_country.fit_transform(d_model['country'])
    d_model['continent_encoded'] = le_continent.fit_transform(d_model['continent'])
    
    # Sort by date
    d_model = d_model.sort_values('date_time')
    
    # Features: use current data to predict next
    features = ['latitude', 'longitude', 'depth', 'magnitude', 
                'year', 'month', 'day', 'hour', 'timestamp',
                'country_encoded', 'continent_encoded']
    
    X = d_model[features].iloc[:-1].values
    
    # Targets: next event
    y_lat = d_model['latitude'].iloc[1:].values
    y_lon = d_model['longitude'].iloc[1:].values
    y_mag = d_model['magnitude'].iloc[1:].values
    
    # Split train/test
    test_size = min(0.2, 10 / len(X))
    X_train, X_test, y_lat_train, y_lat_test = train_test_split(X, y_lat, test_size=test_size, random_state=42)
    _, _, y_lon_train, y_lon_test = train_test_split(X, y_lon, test_size=test_size, random_state=42)
    _, _, y_mag_train, y_mag_test = train_test_split(X, y_mag, test_size=test_size, random_state=42)
    
    # Train models
    model_lat = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    model_lon = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    model_mag = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    
    model_lat.fit(X_train, y_lat_train)
    model_lon.fit(X_train, y_lon_train)
    model_mag.fit(X_train, y_mag_train)
    
    # Prediction: use the last known event
    last_event = X[-1].reshape(1, -1)
    
    pred_lat = model_lat.predict(last_event)[0]
    pred_lon = model_lon.predict(last_event)[0]
    pred_mag = model_mag.predict(last_event)[0]
    
    # Scores
    score_lat = model_lat.score(X_test, y_lat_test)
    score_lon = model_lon.score(X_test, y_lon_test)
    score_mag = model_mag.score(X_test, y_mag_test)
    
    # Results
    results = dbc.Card([
        dbc.CardBody([
            html.H5("Next Earthquake Prediction", className='mb-3'),
            html.P([
                html.Strong("Predicted Location: "),
                f"Lat: {pred_lat:.2f}°, Lon: {pred_lon:.2f}°"
            ]),
            html.P([
                html.Strong("Predicted Magnitude: "),
                f"{pred_mag:.2f}"
            ]),
            html.Hr(),
            html.H6("Model Accuracy (R² Score):"),
            html.Ul([
                html.Li(f"Latitude: {score_lat:.3f}"),
                html.Li(f"Longitude: {score_lon:.3f}"),
                html.Li(f"Magnitude: {score_mag:.3f}")
            ]),
            html.Small("Note: Model trained with Random Forest. This is a prototype, for better prediction more rigorous methods would be needed, such as choosing an LSTM that takes into account the temporal sequence of data.", 
                      className='text-muted')
        ])
    ], className='mb-3')
    
    # Prediction map
    fig = go.Figure()
    
    # Historical earthquakes (last 100)
    recent = d_model.tail(100)
    fig.add_trace(go.Scattergeo(
        lat=recent['latitude'],
        lon=recent['longitude'],
        mode='markers',
        marker=dict(
            size=recent['magnitude'] * 2,
            color='lightblue',
            opacity=0.5,
            line=dict(width=0.5, color='white')
        ),
        name='Historical',
        text=recent['magnitude'],
        hovertemplate='Mag: %{text:.1f}<extra></extra>'
    ))
    
    # Prediction
    fig.add_trace(go.Scattergeo(
        lat=[pred_lat],
        lon=[pred_lon],
        mode='markers',
        marker=dict(
            size=pred_mag * 4,
            color='red',
            symbol='star',
            line=dict(width=2, color='darkred')
        ),
        name='Prediction',
        text=[pred_mag],
        hovertemplate='Prediction<br>Mag: %{text:.1f}<extra></extra>'
    ))
    
    fig.update_layout(
        geo=dict(
            projection_type='natural earth', #orthographic
            showland=True,
            landcolor='rgb(243, 243, 243)',
            coastlinecolor='rgb(204, 204, 204)',
            center=dict(lat=pred_lat, lon=pred_lon),
            projection_scale=2
        ),
        height=400,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )
    
    return results, fig

# -------------------------
# Callback para resetear filtros
# -------------------------
@app.callback(
    [Output('filter-continent', 'value'), Output('filter-country', 'value'), Output('filter-year', 'value'), Output('filter-tsunami', 'value'), Output('filter-alerts', 'value')],
    Input('reset-filters', 'n_clicks')
)
def reset_filters(n):
    return 'All', 'All', [max(YEAR_MIN, YEAR_MAX - 4), YEAR_MAX], [], ['All']

# -------------------------
# Ejecutar la aplicación
# -------------------------

if __name__ == '__main__':
    if HOSTING:
        port = int(os.environ.get('PORT', 8050))
        app.run(debug=False, host='0.0.0.0', port=port)
    else:
        app.run(debug=True, port=8050) 


ConnectionError: HTTPConnectionPool(host='0.0.0.0', port=8050): Max retries exceeded with url: /_alive_6e34b7f0-3b19-4403-ba6d-3c361dfd3aa0 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000218931F3170>: Failed to establish a new connection: [WinError 10049] La dirección solicitada no es válida en este contexto'))